# Motor de Simulación Salarial
### Preprocesamiento de Datos — Actividades 1 al 5
**Estudiantes:** Breyder David Cabrera · Jessica Malpud  
**Dataset:** Global AI Jobs & Skills Demand Dataset (2020–2026)  
**Fuente:** https://www.kaggle.com/datasets/chaitanyajamble/global-ai-jobs-and-skills-demand-dataset-20202026/data

## Importaciones

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Carga del dataset

In [ ]:
df = pd.read_csv('global_ai_jobs_dataset.csv')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head()

## Actividad 1 — Verificación y corrección de tipos de datos
Se identifican los tipos de cada variable y se corrigen los incorrectos:
- `job_id`: se elimina (identificador sin valor predictivo)
- `posting_date`: se convierte de `str` a `datetime64`
- El resto de variables tienen tipos correctos

In [ ]:
print('=== Tipos originales ===')
print(df.dtypes)
print()

# job_id no aporta información predictiva
df = df.drop(columns=['job_id'])

# posting_date: string → datetime
df['posting_date'] = pd.to_datetime(df['posting_date'])

print('=== Tipos corregidos ===')
print(df.dtypes)
print()
print('=== Valores nulos ===')
print(df.isnull().sum())

## Actividad 2 — Partición Train / Test
Se divide el dataset en 80% entrenamiento y 20% prueba usando `train_test_split` de scikit-learn.  
Se usa `stratify=experience_level` para mantener la proporción de niveles de experiencia en ambos conjuntos.

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['experience_level']
)

print(f'Train: {len(train_df):,} filas ({len(train_df)/len(df):.0%})')
print(f'Test:  {len(test_df):,} filas ({len(test_df)/len(df):.0%})')
print()
print('Distribución de experience_level en train:')
print(train_df['experience_level'].value_counts(normalize=True).round(3))
print()
print('Distribución de experience_level en test:')
print(test_df['experience_level'].value_counts(normalize=True).round(3))

## Actividad 3 — Guardar conjuntos en archivos separados
Se exportan los conjuntos **sin transformar** a `train.csv` y `test.csv`.

In [ ]:
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv',  index=False)

print('Archivos guardados:')
print('  train.csv')
print('  test.csv')

## Actividad 4 — One-Hot Encoding para variables categóricas
Se aplica `pd.get_dummies()` a las 9 variables categóricas nominales.  
De `posting_date` se extraen `posting_year` y `posting_month` como variables numéricas.  
Las columnas `skills` y `tools_used` son listas multi-valor separadas por `;` y requieren tratamiento especial (multi-label encoding), por lo que se excluyen de este paso.  
Se usa `.align()` para garantizar que train y test tengan exactamente las mismas columnas.

In [ ]:
# Extraer año y mes de posting_date
for split in [train_df, test_df]:
    split['posting_year']  = split['posting_date'].dt.year
    split['posting_month'] = split['posting_date'].dt.month

train_df = train_df.drop(columns=['posting_date'])
test_df  = test_df.drop(columns=['posting_date'])

# Variables categóricas a codificar
categorical_cols = [
    'job_title', 'industry', 'company_size', 'country', 'city',
    'experience_level', 'remote_type', 'education_required', 'ai_domain'
]

# Variables texto multi-valor (se excluyen del OHE estándar)
text_cols = ['skills', 'tools_used']

train_enc = pd.get_dummies(train_df.drop(columns=text_cols),
                            columns=categorical_cols, drop_first=False)
test_enc  = pd.get_dummies(test_df.drop(columns=text_cols),
                            columns=categorical_cols, drop_first=False)

# Alinear columnas (test puede tener menos categorías que train)
train_enc, test_enc = train_enc.align(test_enc, join='left', fill_value=0)

print(f'Columnas antes del OHE: 15')
print(f'Columnas después del OHE — train: {train_enc.shape[1]} | test: {test_enc.shape[1]}')
train_enc.head()

## Actividad 5 — Escalamiento de variables numéricas
Se aplica `StandardScaler` (media = 0, desviación estándar = 1) a las variables numéricas continuas.  
**Importante:** el scaler se ajusta (`fit`) únicamente con el conjunto de entrenamiento y luego se aplica (`transform`) al conjunto de prueba para evitar *data leakage*.  
La variable objetivo `salary_usd` **no se escala** en este paso.

In [ ]:
numeric_cols = ['years_experience_required', 'benefits_score',
                'posting_year', 'posting_month']

scaler = StandardScaler()

# fit solo en train, transform en ambos
train_enc[numeric_cols] = scaler.fit_transform(train_enc[numeric_cols])
test_enc[numeric_cols]  = scaler.transform(test_enc[numeric_cols])

print('=== Estadísticos tras escalamiento (train) ===')
train_enc[numeric_cols].describe().round(4)

## Guardar datasets procesados
Se exportan los conjuntos transformados para usar en el entrenamiento del modelo.

In [ ]:
train_enc.to_csv('train_processed.csv', index=False)
test_enc.to_csv('test_processed.csv',  index=False)

print('Archivos guardados:')
print('  train_processed.csv')
print('  test_processed.csv')
print()
print(f'Shape train procesado: {train_enc.shape}')
print(f'Shape test  procesado: {test_enc.shape}')